# RetinaNet Object Detection - Google Colab Training
Notebook để train mô hình RetinaNet trên Google Colab

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Dependencies

In [ ]:
!pip install tensorflow pandas

## 3. Setup Working Directory
**HƯỚNG DẪN:** Upload toàn bộ project lên Google Drive, sau đó thay đổi đường dẫn dưới đây

In [ ]:
import os
import sys

# Thay đổi đường dẫn này theo vị trí project trong Google Drive của bạn
PROJECT_PATH = '/content/drive/MyDrive/de_tai_tot_nghiep'

# Chuyển đến thư mục project
os.chdir(PROJECT_PATH)

# Thêm project vào Python path để import được các module
sys.path.insert(0, PROJECT_PATH)

print(f"Current working directory: {os.getcwd()}")

## 4. Verify GPU

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

# Nếu không có GPU, bạn cần enable GPU trong Runtime > Change runtime type > Hardware accelerator > GPU

## 5. Import Modules

In [ ]:
import tensorflow as tf
from Anchor_box import Anchor_box
from Label_encode import LabelEncoder
from Xu_ly_du_lieu import preprocessing_data_before_training
from RetinaNet import RetinaNet, RetinaNetLoss
from resnet_50 import resnet_50_backbone
import pandas as pd
import ast

## 6. Setup Paths and Parameters

In [ ]:
# Đường dẫn dữ liệu - Thay đổi theo cấu trúc thư mục của bạn
base_dir = os.path.join(PROJECT_PATH, 'object_detect')
train_csv_path = os.path.join(base_dir, 'csv_file/train_data.csv')
val_csv_path = os.path.join(base_dir, 'csv_file/valid_data.csv')

# Tham số
target_size = 224
batch_size = 32
EPOCHS = 100

print(f"Base dir: {base_dir}")
print(f"Train CSV: {train_csv_path}")
print(f"Val CSV: {val_csv_path}")

## 7. Initialize Anchor Box and Label Encoder

In [ ]:
# Khởi tạo Anchor Box
anchor_gene = Anchor_box()
all_anchor = anchor_gene.get_anchors(img_h=target_size, img_w=target_size)

# Khởi tạo Label Encoder
label_encoder = LabelEncoder()

print(f"Anchor boxes shape: {all_anchor.shape}")

## 8. Define Pack Targets Function

In [ ]:
def pack_targets(img, bdb, cls):
    """Gộp target_boxes và target_classes thành y_true"""
    _, target_boxes, target_classes = label_encoder.encode_batch(img, bdb, cls, all_anchor)
    
    # Ép kiểu target_classes về float32
    target_classes = tf.cast(target_classes, tf.float32)
    
    # Mở rộng chiều của target_classes
    target_classes = tf.expand_dims(target_classes, axis=-1)
    
    # Gộp boxes và classes
    y_true = tf.concat([target_boxes, target_classes], axis=-1)
    
    return img, y_true

## 9. Prepare Dataset

In [ ]:
# Đọc và xử lý dữ liệu
raw_train_data = preprocessing_data_before_training.change_string2number(train_csv_path)
raw_val_data = preprocessing_data_before_training.change_string2number(val_csv_path)

# Tạo dataset
train_dataset = preprocessing_data_before_training.create_dataset_from_dataframe(raw_train_data).shuffle(buffer_size=1000)
val_dataset = preprocessing_data_before_training.create_dataset_from_dataframe(raw_val_data)

print("Dataset created successfully!")

## 10. Preprocess Dataset

In [ ]:
# Đọc ảnh và resize
train_dataset = train_dataset.map(
    preprocessing_data_before_training.read_img_and_label,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.map(
    lambda img, bdb, cls: preprocessing_data_before_training.resize_and_pad_img(img, bdb, cls, target_size),
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    preprocessing_data_before_training.read_img_and_label,
    num_parallel_calls=tf.data.AUTOTUNE
)
val_dataset = val_dataset.map(
    lambda img, bdb, cls: preprocessing_data_before_training.resize_and_pad_img(img, bdb, cls, target_size),
    num_parallel_calls=tf.data.AUTOTUNE
)

print("Preprocessing done!")

## 11. Batch and Pack Targets

In [ ]:
# Batch dataset
train_dataset = train_dataset.padded_batch(
    batch_size=batch_size,
    padding_values=(0.0, 1e-8, -2),
    drop_remainder=True
)
train_dataset = train_dataset.map(pack_targets, num_parallel_calls=tf.data.AUTOTUNE)

val_dataset = val_dataset.padded_batch(
    batch_size=batch_size,
    padding_values=(0.0, 1e-8, -2),
    drop_remainder=True
)
val_dataset = val_dataset.map(pack_targets, num_parallel_calls=tf.data.AUTOTUNE)

# Prefetch
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

print("Dataset ready for training!")

## 12. Calculate Number of Classes

In [ ]:
# Đọc số lượng classes từ train data
train_df = pd.read_csv(train_csv_path)
all_labels = []
for cid in train_df['class_id'].apply(ast.literal_eval):
    all_labels.extend(cid)

num_classes = len(set(all_labels))
print(f"Number of classes: {num_classes}")

## 13. Build Model

In [ ]:
# Khởi tạo backbone và model
resnet50_backbone = resnet_50_backbone()
model = RetinaNet(num_classes=num_classes, backbone=resnet50_backbone)

print("Model created successfully!")

## 14. Setup Optimizer and Loss

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
loss_fn = RetinaNetLoss(num_classes=num_classes)

# 3. Compile model
model.compile(optimizer=optimizer, loss=loss_fn)
print("Model compiled!")

## 15. Setup Callbacks

In [ ]:
weight_folder = os.path.join(base_dir, 'weight_store')
os.makedirs(weight_folder, exist_ok=True)

callbacks_list = [

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="loss",  # Theo doi gia tri loss
            factor=0.5,  # Neu khong giam thi cắt đôi LR (ví dụ 1e-4 -> 5e-5)
            patience=5,  # Dung nhu bro muon: 5 lan khong doi thi giam
            min_lr=1e-7,  # Khong cho giam sau qua muc nay
            verbose=1  # In thong bao moi khi giam de bro biet
        ),

        tf.keras.callbacks.ModelCheckpoint(
            filepath=weight_folder + "/test_256_adam_auto.weights.h5",
            monitor="loss",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="loss",
            patience=15,
            verbose=1,
            restore_best_weights=True
        )
    ]

print(f"Weights will be saved to: {weight_folder}")

## 16. Train Model

In [ ]:
# Training
hist = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks_list,
)

print("Training completed!")

## 17. Save Training History

In [ ]:
# Lưu history
hist_df = pd.DataFrame(hist.history)

hist_store_folder = os.path.join(base_dir, 'hist_store')
os.makedirs(hist_store_folder, exist_ok=True)

hist_csv_path = os.path.join(hist_store_folder, '100epochs_adam_csv.csv')
hist_df.to_csv(hist_csv_path)

print(f"Training history saved to: {hist_csv_path}")

## 18. Plot Training History (Optional)

In [ ]:
import matplotlib.pyplot as plt

# Plot loss
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(hist.history['loss'], label='Train Loss')
if 'val_loss' in hist.history:
    plt.plot(hist.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')

plt.tight_layout()
plt.show()